# DSatur: Graph Colouring by Saturation Degree

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
from collections import defaultdict
import math

matplotlib.rcParams['font.family'] = 'monospace'
matplotlib.rcParams['font.size'] = 11

# ---------------------------------------------------------------------------
# Graph definition
# ---------------------------------------------------------------------------
VERTICES = ['A', 'B', 'C', 'D', 'E']
EDGES = [('A', 'B'), ('A', 'D'), ('A', 'E'),
         ('B', 'C'), ('B', 'D'),
         ('C', 'D'),
         ('D', 'E')]

ADJ = defaultdict(set)
for u, v in EDGES:
    ADJ[u].add(v)
    ADJ[v].add(u)

# Circular layout (fixed positions)
N = len(VERTICES)
POS = {}
for i, v in enumerate(VERTICES):
    angle = 2 * math.pi * i / N - math.pi / 2  # start from top
    POS[v] = (math.cos(angle), math.sin(angle))

# Pleasant colour palette (up to 6 colours)
PALETTE = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B3', '#CCB974']
UNCOLOURED = '#D3D3D3'

# Known chromatic number for this graph
CHROMATIC_NUMBER = 4  # K4 minor (A-B-C-D forms a 4-clique via edges)


# ---------------------------------------------------------------------------
# Algorithm implementations
# ---------------------------------------------------------------------------
def dsatur_order(vertices, adj):
    """Return the sequence of (vertex, colour) assignments using DSatur."""
    colour = {}  # vertex -> colour index (0-based)
    steps = []   # list of snapshots

    remaining = set(vertices)

    while remaining:
        # Compute saturation and uncoloured degree for every remaining vertex
        sat = {}
        udeg = {}
        for v in remaining:
            neighbour_colours = {colour[u] for u in adj[v] if u in colour}
            sat[v] = len(neighbour_colours)
            udeg[v] = sum(1 for u in adj[v] if u in remaining and u != v)

        # Pick vertex with highest saturation, break ties by uncoloured degree,
        # then alphabetical
        chosen = max(remaining, key=lambda v: (sat[v], udeg[v], v))

        # Assign smallest available colour
        used = {colour[u] for u in adj[chosen] if u in colour}
        c = 0
        while c in used:
            c += 1
        colour[chosen] = c
        remaining.remove(chosen)

        # Save snapshot
        steps.append({
            'chosen': chosen,
            'colour': dict(colour),
            'sat': {v: sat.get(v, 0) for v in vertices},
            'udeg': {v: udeg.get(v, 0) for v in vertices},
        })

    return steps


def max_degree_order(vertices, adj):
    """Greedy colouring using static max-degree ordering."""
    order = sorted(vertices, key=lambda v: (-len(adj[v]), v))
    colour = {}
    steps = []

    for chosen in order:
        remaining_before = set(vertices) - set(colour) - {chosen}
        sat = {}
        udeg = {}
        for v in (remaining_before | {chosen}):
            neighbour_colours = {colour[u] for u in adj[v] if u in colour}
            sat[v] = len(neighbour_colours)
            udeg[v] = sum(1 for u in adj[v] if u not in colour and u != v)

        used = {colour[u] for u in adj[chosen] if u in colour}
        c = 0
        while c in used:
            c += 1
        colour[chosen] = c

        # Fill in sat/udeg for already-coloured vertices too
        full_sat = {}
        full_udeg = {}
        for v in vertices:
            neighbour_colours = {colour[u] for u in adj[v] if u in colour}
            full_sat[v] = len(neighbour_colours)
            full_udeg[v] = sum(1 for u in adj[v] if u not in colour)

        steps.append({
            'chosen': chosen,
            'colour': dict(colour),
            'sat': full_sat,
            'udeg': full_udeg,
        })

    return steps


# ---------------------------------------------------------------------------
# Drawing
# ---------------------------------------------------------------------------
def draw_state(ax, step_data, vertices, edges, pos, palette, highlight):
    """Draw the graph on the given axes."""
    ax.set_xlim(-1.55, 1.55)
    ax.set_ylim(-1.55, 1.55)
    ax.set_aspect('equal')
    ax.axis('off')

    colour_map = step_data['colour']

    # Draw edges
    for u, v in edges:
        xu, yu = pos[u]
        xv, yv = pos[v]
        ax.plot([xu, xv], [yu, yv], '-', color='#888888', linewidth=1.5, zorder=1)

    # Draw vertices
    for v in vertices:
        x, y = pos[v]
        c_idx = colour_map.get(v)
        face = palette[c_idx] if c_idx is not None else UNCOLOURED
        lw = 3.5 if v == highlight else 1.2
        ec = '#222222' if v == highlight else '#555555'
        circle = plt.Circle((x, y), 0.15, facecolor=face, edgecolor=ec,
                            linewidth=lw, zorder=3)
        ax.add_patch(circle)
        ax.text(x, y, v, ha='center', va='center', fontsize=13,
                fontweight='bold', color='white' if c_idx is not None else '#333',
                zorder=4, family='monospace')


def draw_table(ax, step_data, vertices, palette):
    """Draw the info table on the given axes."""
    ax.axis('off')
    colour_map = step_data['colour']
    sat = step_data['sat']
    udeg = step_data['udeg']
    chosen = step_data['chosen']

    headers = ['Vtx', 'Sat', 'UDeg', 'Colour']
    col_x = [0.05, 0.25, 0.50, 0.75]
    top = 0.92
    row_h = 0.13

    for j, h in enumerate(headers):
        ax.text(col_x[j], top, h, fontsize=11, fontweight='bold',
                family='monospace', va='center')

    ax.plot([0.0, 0.95], [top - 0.05, top - 0.05], '-', color='#aaa', lw=0.8)

    for i, v in enumerate(vertices):
        y = top - (i + 1) * row_h
        weight = 'bold' if v == chosen else 'normal'
        prefix = '>' if v == chosen else ' '
        ax.text(col_x[0], y, f'{prefix}{v}', fontsize=11, fontweight=weight,
                family='monospace', va='center')
        ax.text(col_x[1], y, str(sat.get(v, '-')), fontsize=11,
                fontweight=weight, family='monospace', va='center')
        ax.text(col_x[2], y, str(udeg.get(v, '-')), fontsize=11,
                fontweight=weight, family='monospace', va='center')

        c_idx = colour_map.get(v)
        if c_idx is not None:
            label = f'C{c_idx + 1}'
            colour = palette[c_idx]
        else:
            label = '--'
            colour = '#999999'
        ax.text(col_x[3], y, label, fontsize=11, fontweight=weight,
                family='monospace', va='center', color=colour)


def draw_summary(ax, steps, palette):
    """Draw summary after the last step."""
    final_colours = steps[-1]['colour']
    n_colours = len(set(final_colours.values()))
    optimal = n_colours == CHROMATIC_NUMBER
    msg = (f'Colours used: {n_colours}\n'
           f'Chromatic number: {CHROMATIC_NUMBER}\n'
           f'{"Optimal!" if optimal else "Not optimal."}')
    ax.text(0.05, 0.08, msg, fontsize=11, family='monospace',
            va='bottom', linespacing=1.6,
            color='#2a7f2a' if optimal else '#b03030')

In [ ]:
# ---------------------------------------------------------------------------
# Interactive widget
# ---------------------------------------------------------------------------

strategy_dropdown = widgets.Dropdown(
    options=[('DSatur', 'dsatur'), ('Max Degree', 'maxdeg')],
    value='dsatur',
    description='Strategy:',
    style={'description_width': 'auto'},
)

btn_first = widgets.Button(description='|<< First', layout=widgets.Layout(width='90px'))
btn_prev  = widgets.Button(description='< Prev',    layout=widgets.Layout(width='90px'))
btn_next  = widgets.Button(description='Next >',    layout=widgets.Layout(width='90px'))
btn_last  = widgets.Button(description='Last >>|',  layout=widgets.Layout(width='90px'))
step_label = widgets.Label(value='Step 1 / 5')

out = widgets.Output()

state = {'step': 0, 'steps': []}


def compute_steps():
    if strategy_dropdown.value == 'dsatur':
        state['steps'] = dsatur_order(VERTICES, ADJ)
    else:
        state['steps'] = max_degree_order(VERTICES, ADJ)
    state['step'] = 0


def render():
    steps = state['steps']
    idx = state['step']
    step_label.value = f'Step {idx + 1} / {len(steps)}'

    with out:
        clear_output(wait=True)
        fig, (ax_graph, ax_table) = plt.subplots(
            1, 2, figsize=(10, 4.5),
            gridspec_kw={'width_ratios': [1.3, 1]}
        )
        fig.subplots_adjust(wspace=0.05)

        step_data = steps[idx]
        strategy_name = 'DSatur' if strategy_dropdown.value == 'dsatur' else 'Max Degree'
        ax_graph.set_title(
            f'{strategy_name}  \u2014  colouring vertex {step_data["chosen"]}',
            fontsize=12, family='monospace', pad=10
        )

        draw_state(ax_graph, step_data, VERTICES, EDGES, POS, PALETTE,
                   highlight=step_data['chosen'])
        draw_table(ax_table, step_data, VERTICES, PALETTE)

        if idx == len(steps) - 1:
            draw_summary(ax_table, steps, PALETTE)

        plt.show()


def on_first(_):
    state['step'] = 0
    render()

def on_prev(_):
    state['step'] = max(0, state['step'] - 1)
    render()

def on_next(_):
    state['step'] = min(len(state['steps']) - 1, state['step'] + 1)
    render()

def on_last(_):
    state['step'] = len(state['steps']) - 1
    render()

def on_strategy_change(_):
    compute_steps()
    render()


btn_first.on_click(on_first)
btn_prev.on_click(on_prev)
btn_next.on_click(on_next)
btn_last.on_click(on_last)
strategy_dropdown.observe(on_strategy_change, names='value')

controls = widgets.HBox(
    [strategy_dropdown, btn_first, btn_prev, step_label, btn_next, btn_last],
    layout=widgets.Layout(gap='6px', align_items='center')
)

compute_steps()
display(widgets.VBox([controls, out]))
render()